# 01 -- Config layer

`qwenmoe/config.py`'s `ModelConfig` / `TrainConfig` / `DataConfig`
dataclasses are this project's single source of truth for every tunable
knob -- no YAML/JSON schema layer, presets are just plain Python
(`configs/presets.py`). Each one validates itself in `__post_init__`,
raising `ValueError` eagerly on an invalid combination rather than failing
confusingly deep inside a model/training run.

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## `ModelConfig`
| field | default | meaning |
|---|---|---|
| `vocab_size` | required | size of the token vocabulary |
| `d_model` | 256 | residual-stream width |
| `n_layers` | 6 | number of `TransformerBlock`s |
| `n_heads` / `n_kv_heads` | 8 / 4 | query heads / key-value heads (GQA; equal = plain MHA) |
| `d_ff` | 512 | per-expert feed-forward width |
| `n_shared_experts` / `n_routed_experts` / `top_k` | 1 / 8 / 2 | MoE sizing |
| `positional_scheme` | "rope" | "rope" \| "nope" \| "alibi" |
| `activation` | "swiglu" | "swiglu" \| "geglu" \| "gelu" |
| `moe_balancing` | "aux_loss" | "aux_loss" \| "loss_free" |
| `attention_variant` | "gqa" | "gqa" \| "mla" |
| `mla_latent_dim` | None | required when `attention_variant="mla"` |
| `tie_word_embeddings` | True | share the LM head weight with the embedding table |

In [2]:
from qwenmoe.config import ModelConfig

config = ModelConfig(vocab_size=65, d_model=64, n_heads=4, n_kv_heads=2, n_routed_experts=4, top_k=2)
config

ModelConfig(vocab_size=65, d_model=64, n_layers=6, n_heads=4, n_kv_heads=2, d_ff=512, n_shared_experts=1, n_routed_experts=4, top_k=2, max_seq_len=256, positional_scheme='rope', activation='swiglu', moe_balancing='aux_loss', attention_variant='gqa', mla_latent_dim=None, lora_rank=None, tie_word_embeddings=True, rope_theta=10000.0)

### Validation demo
`__post_init__` catches invalid combinations immediately, with a message
that says exactly what's wrong -- not a confusing shape-mismatch error
several layers deep inside a forward pass.

In [3]:
invalid_configs = [
    dict(vocab_size=0),
    dict(vocab_size=10, d_model=10, n_heads=3),               # d_model not divisible by n_heads
    dict(vocab_size=10, n_heads=4, n_kv_heads=3),              # n_heads not divisible by n_kv_heads
    dict(vocab_size=10, top_k=5, n_routed_experts=2),          # top_k > n_routed_experts
    dict(vocab_size=10, attention_variant="mla"),              # mla_latent_dim missing
    dict(vocab_size=10, attention_variant="mla", mla_latent_dim=0),
]
for kwargs in invalid_configs:
    try:
        ModelConfig(**kwargs)
        print("NO ERROR (unexpected) for", kwargs)
    except ValueError as e:
        print(f"{kwargs} ->\n   {e}")

{'vocab_size': 0} ->
   vocab_size must be >= 1, got 0
{'vocab_size': 10, 'd_model': 10, 'n_heads': 3} ->
   d_model (10) must be divisible by n_heads (3)
{'vocab_size': 10, 'n_heads': 4, 'n_kv_heads': 3} ->
   n_heads (4) must be divisible by n_kv_heads (3)
{'vocab_size': 10, 'top_k': 5, 'n_routed_experts': 2} ->
   top_k (5) must be between 1 and n_routed_experts (2)
{'vocab_size': 10, 'attention_variant': 'mla'} ->
   mla_latent_dim is required when attention_variant == 'mla'
{'vocab_size': 10, 'attention_variant': 'mla', 'mla_latent_dim': 0} ->
   mla_latent_dim must be >= 1, got 0


## `TrainConfig`
`batch_size`, `seq_len`, `total_tokens` (the scaling-law-derived budget,
see notebook 05) are required; `lr`, `warmup_steps`,
`checkpoint_interval_steps`, `eval_interval_steps`, `max_eval_batches`,
`log_interval_steps`, `device`, `seed` all have sensible defaults.

In [4]:
from qwenmoe.config import TrainConfig

train_config = TrainConfig(batch_size=16, seq_len=32, total_tokens=2_000_000)
print(train_config)

for bad_kwargs in [
    dict(batch_size=0, seq_len=32, total_tokens=1000),
    dict(batch_size=1, seq_len=0, total_tokens=1000),
    dict(batch_size=1, seq_len=1, total_tokens=0),
]:
    try:
        TrainConfig(**bad_kwargs)
    except ValueError as e:
        print(bad_kwargs, "->", e)

TrainConfig(batch_size=16, seq_len=32, total_tokens=2000000, lr=0.0003, warmup_steps=200, checkpoint_interval_steps=500, eval_interval_steps=100, max_eval_batches=50, log_interval_steps=10, device='auto', seed=1337)
{'batch_size': 0, 'seq_len': 32, 'total_tokens': 1000} -> batch_size must be >= 1, got 0
{'batch_size': 1, 'seq_len': 0, 'total_tokens': 1000} -> seq_len must be >= 1, got 0
{'batch_size': 1, 'seq_len': 1, 'total_tokens': 0} -> total_tokens must be >= 1, got 0


## `DataConfig`
`corpus` and `tokenizer_type` are required; `cache_dir` is coerced from a
plain string into a real `Path` automatically. `fixed_slice_tokens` turns
on the "ablation mode" a comparison run needs (notebook 05).

In [5]:
from qwenmoe.config import DataConfig
from pathlib import Path

data_config = DataCo   nfig(corpus="tiny_shakespeare", tokenizer_type="char", cache_dir=".cache/data")
print(data_config)
print("cache_dir became a real Path:", isinstance(data_config.cache_dir, Path))

try:
    DataConfig(corpus="tiny_shakespeare", tokenizer_type="bpe", bpe_vocab_size=0)
except ValueError as e:
    print("bad bpe_vocab_size ->", e)

DataConfig(corpus='tiny_shakespeare', tokenizer_type='char', bpe_vocab_size=4096, cache_dir=PosixPath('.cache/data'), fixed_slice_tokens=None, fixed_slice_seed=0)
cache_dir became a real Path: True
bad bpe_vocab_size -> bpe_vocab_size must be >= 1, got 0


## `configs/presets.py` -- config as code
`BASELINE` (the illustrative full-size config) and `SMOKE_TEST` (tiny,
runs end to end in minutes) are hand-written; the 5 ablation presets
(`POS_NOPE`, `POS_ALIBI`, `ACT_GEGLU`, `ACT_GELU`, `MOE_LOSSFREE`) are each
a one-field `dataclasses.replace(SMOKE_TEST, ...)`.

In [6]:
import dataclasses
from configs import presets

for name in ["BASELINE", "SMOKE_TEST", "POS_NOPE", "POS_ALIBI", "ACT_GEGLU", "ACT_GELU", "MOE_LOSSFREE"]:
    cfg = getattr(presets, name)
    print(f"{name:12s} d_model={cfg.d_model:4d}  n_layers={cfg.n_layers}  "
          f"positional={cfg.positional_scheme:6s}  activation={cfg.activation:7s}  balancing={cfg.moe_balancing}")

print()
print("every ablation preset really is a one-field dataclasses.replace(SMOKE_TEST, ...):")
print(presets.POS_NOPE == dataclasses.replace(presets.SMOKE_TEST, positional_scheme="nope"))

BASELINE     d_model= 256  n_layers=6  positional=rope    activation=swiglu   balancing=aux_loss
SMOKE_TEST   d_model=  64  n_layers=2  positional=rope    activation=swiglu   balancing=aux_loss
POS_NOPE     d_model=  64  n_layers=2  positional=nope    activation=swiglu   balancing=aux_loss
POS_ALIBI    d_model=  64  n_layers=2  positional=alibi   activation=swiglu   balancing=aux_loss
ACT_GEGLU    d_model=  64  n_layers=2  positional=rope    activation=geglu    balancing=aux_loss
ACT_GELU     d_model=  64  n_layers=2  positional=rope    activation=gelu     balancing=aux_loss
MOE_LOSSFREE d_model=  64  n_layers=2  positional=rope    activation=swiglu   balancing=loss_free

every ablation preset really is a one-field dataclasses.replace(SMOKE_TEST, ...):
True
